# Enter all dependencies here

In [1]:
import numpy as np
from scipy.stats import lognorm
import matplotlib.pyplot as plt
#plt.rcParams['text.usetex'] = True
plt.rcParams['font.size'] = 16
from datetime import datetime
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

# Function to read in Cosmic Watch (CW) data

In [ ]:
####
#### read in CW data file
####
def getCosmicWatch(fname,_BINSIZE_=60,_TIMEOFFSET_=0):
    # output file:
    # [count] [time stamp] [ADC] [SiPM] [deadtime] [temp]
    count,time,adc,sipm,deadt,temp = np.genfromtxt(fname=fname,delimiter=' ',skip_header=6,unpack=True)

    ####
    #### The following code can be used to select certain energies for the recorded events
    ####
    ####
    #### a gaussian function (for fitting)
    ####
    # def gaussian(x, mean, amp, std, c):
    #     return amp * np.exp( - (x-mean)**2 / (2*std**2)) + c

    # #### find the peak (most common) voltage for all events
    # #n,bins,patches = plt.hist(sipm,bins=250)
    # n, bins = np.histogram(sipm,250)
    # bins = bins[:-1] + np.diff(bins) /2
    # param,covar = curve_fit(gaussian,bins,n,p0=[50.,np.max(n),10.,np.average(n[-10:-1])])
    # #print(param,np.sqrt(np.diag(covar)))
    # #print(param[0],param[2])

    # #### this mask could be used, e.g., to select
    # #### high energy events, low energy events, etc.
    # mask = adc>0.

    # count = count[mask]
    # time = time[mask]
    # adc = adc[mask]
    # sipm = sipm[mask]
    # deadt = deadt[mask]
    # temp = temp[mask]

    ####
    #### calculate the count rate in chunks of _BINSIZE_ (above)
    ####
    event_deadt = np.diff(np.append([0],deadt))
    # total_time = max(time)
    # total_deadt = max(deadt)
    # #total_counts = max(count)-min(count)
    # total_counts = len(count)

    ## change time from milliseconds to seconds
    time_s = time/1000.
    event_deadt_s = event_deadt/1000.
    #total_time_s = total_time/1000.
    #total_deadt_s = total_deadt/1000.
    event_livetime_s = np.diff(np.append([0],time_s)) - event_deadt_s

    ####
    #### determine the overall count rate
    ####
    # live_time = (total_time - total_deadt)/1000.
    # weights = np.ones(len(count))/live_time
    # count_rate = total_counts/live_time
    # count_rate_err = np.sqrt(total_counts)/live_time
    # ## print the overall count rate
    # print('Count rate: ',count_rate,' +/- ',count_rate_err)

    ####
    #### bin the data for nice plots
    ####
    bins = range(int(min(time_s)),int(max(time_s)),_BINSIZE_)
    counts,binEdges = np.histogram(time_s, bins=bins)
    bin_livetime,binEdges = np.histogram(time_s,bins=bins,weights=event_livetime_s)

    bincenters = 0.5*(binEdges[1:]+binEdges[:-1])
    #binned_counts = counts
    #binned_counts_err = np.sqrt(counts)
    binned_count_rate = counts/bin_livetime
    binned_count_rate_err = np.sqrt(counts)/bin_livetime
    binned_time = bincenters/_BINSIZE_

    return binned_time+_TIMEOFFSET_,binned_count_rate,binned_count_rate_err

# Function to read in Geiger Counter data

In [ ]:
####
#### get data from Geiger counter (GC)
####
def getGeiger(fname,_TIMEOFFSET_=300):
    with open(fname) as f:
        lines = f.readlines()
        START = datetime.strptime(lines[2].rstrip().split(',')[0],'%m/%d/%Y %H:%M')
        time = [0]
        rate = [0]
        for line in lines[3:]:
            line = line.rstrip()
            parse = line.split(',')
            NOW = datetime.strptime(parse[0],"%m/%d/%Y %H:%M")
            time.append((NOW-START).total_seconds()+_TIMEOFFSET_)
            rate.append(int(parse[3]))

    return np.array(time),np.array(rate),np.array(np.sqrt(rate))

# Function to read in Flight Computer information

In [ ]:
####
#### Read in flight computer (FC) data
####
def getFlightComputer(fname):
    timestamp,hr,min,sec,altitude,speed,temp,press = np.genfromtxt(fname,delimiter=',',usecols=(0,2,3,4,5,6,11,12),unpack=True,skip_header=33)
    goodMask = altitude>0
    hr = hr[goodMask]
    min = min[goodMask]
    sec = sec[goodMask]
    altitude = altitude[goodMask]
    speed = speed[goodMask]
    temp = temp[goodMask]
    press = press[goodMask]

    ## get rid of bad/duplicate entries
    uAlt, altInd = np.unique(altitude,return_index=True)
    mask = np.sort(altInd)
    uTime = timestamp[mask]
    uHr = hr[mask]
    uMin = min[mask]
    uSec = sec[mask]
    uAlt = altitude[mask]
    uSpeed = speed[mask]
    uTemp = temp[mask]
    uPress = press[mask]

    ## find flight time in seconds
    t0 = uHr[0]*3600. + uMin[0]*60. + uSec[0]
    flightTime = (uHr*3600. + uMin*60. + uSec) - t0

    return uAlt,flightTime,uSpeed,uTemp,uPress

# Function to read in APRS data

In [ ]:
####
#### Get data from the APRS trackers
####
def getAPRS(fname,type='K'):
    with open(fname) as f:
        lines = f.readlines()
        START = datetime.strptime(lines[1].rstrip().split(',')[0],"%Y-%m-%d %H:%M:%S")
        time = [0]
        speed = [int(lines[1].rstrip().split(',')[4])]
        altitude = [float(lines[1].rstrip().split(',')[6])]
        if(type == 'K'):
            temp = [int(lines[1].rstrip().split(',')[7].split(' ')[5][0:-1])]
            press = [int(lines[1].rstrip().split(',')[7].split(' ')[6][0:-2])/100.] ## convert pressure to hPa
        else:
            temp = [float(lines[1].rstrip().split(',')[7].split(' ')[3][0:-1])]
            press = [int(lines[1].rstrip().split(',')[7].split(' ')[4][0:-3])/100.] ## convert pressure to hPa
        for line in lines[2:]:
            line = line.rstrip()
            parse = line.split(',')
            NOW = datetime.strptime(parse[0],"%Y-%m-%d %H:%M:%S")
            try:
                altitude.append(float(parse[6]))
            except:
                continue
            speed.append(int(parse[4]))
            time.append((NOW-START).total_seconds())

            data = parse[7].split(' ')
            if(type == 'K'):
                temp.append(int(data[5][0:-1]))
                press.append(int(data[6][0:-2])/100.) ## convert pressure to hPa
            else:
                temp.append(float(data[3][0:-1]))
                press.append(int(data[4][0:-3])/100.) ## convert pressure to hPa
    return np.array(time),np.array(speed),np.array(altitude),np.array(temp),np.array(press)

# Code to get and display all information

In [ ]:
######## some constants

#### how many seconds do you want to bin in the plot for the CosmicWatch (CW)?
_BINSIZE_ = 60
#### to normalize by detector area
## CW: 5x5 cm
## GC: 7x1 cm
_CW_AREA_ = 5.*5.
_GC_AREA_ = 7.*1.
#### the order of the fit to the count rate
_FITORDER_ = 7
####
_CWTIMEOFFSET_ = -8.7 ## for syncing between GC, CW, & FC
_GCTIMEOFFSET_ = 300 ## for syncing between GC, CW, & FC

##########################################

####
#### read in CW data file
####
binned_time,binned_count_rate,binned_count_rate_err = getCosmicWatch("/content/drive/MyDrive/Research/Students/HERA/May3-Launch/FILEM001.TXT",_BINSIZE_,_CWTIMEOFFSET_)

##########################################

####
#### Read in flight computer (FC) data
####
uAlt,flightTime,uSpeed,uTemp,uPress = getFlightComputer('/content/drive/MyDrive/Research/Students/HERA/May3-Launch/sensorData05.txt')

####
#### Make a model of the flight (altitude versus time)
####
## ascent: 2800 - 10332
ascMask = ((flightTime>2800)&(flightTime<10332))
## descent: 10332 - 12800
desMask = ((flightTime>10332)&(flightTime<12800))

## fit the data
ascFit = np.polyfit(flightTime[ascMask],uAlt[ascMask],5)
ascModel = np.poly1d(ascFit)
desFit = np.polyfit(flightTime[desMask],uAlt[desMask],5)
desModel = np.poly1d(desFit)

##########################################


####
#### Get data from the APRS trackers
####
tK5,sK5,aK5,cK5,pK5 = getAPRS("/content/drive/MyDrive/Research/Students/HERA/K5ASU-11_CLEAN.csv")
tW5,sW5,aW5,cW5,pW5 = getAPRS("/content/drive/MyDrive/Research/Students/HERA/W5QX-11_CLEAN.csv",'W')

####
#### Make a model of the flight (altitude versus time)
####
## burst at 7450
## flight computer burst at 10332
## DIFF: 2882
tK5 += 2882
tW5 += 2882
## fit the data
ascFit_K = np.polyfit(tK5[tK5<10332],aK5[tK5<10332],7)
ascModel_K = np.poly1d(ascFit_K)
ascFit_W = np.polyfit(tW5[tW5<10332],aW5[tW5<10332],7)
ascModel_W = np.poly1d(ascFit_W)

desFit_K = np.polyfit(tK5[tK5>10332],aK5[tK5>10332],7)
desModel_K = np.poly1d(desFit_K)
desFit_W = np.polyfit(tW5[tW5>10332],aW5[tW5>10332],7)
desModel_W = np.poly1d(desFit_W)

##########################################

####
#### get data from Geiger counter (GC)
####
gTime,gRate,gErr = getGeiger('/content/drive/MyDrive/Research/Students/HERA/May3-Launch/geigerData.csv',_GCTIMEOFFSET_)

##########################################
######## Make some plots
##########################################

####
#### plot the temperature profile
####
# plt.scatter(uAlt/1000,uTemp,marker='o',label="Flight Computer (inside box)")
# plt.scatter(aW5/1000,cW5,marker='o',label="APRS (inside box)")
# plt.scatter(aK5/1000,cK5,marker='o',label="APRS (outside box)")
# plt.xlabel("Altitude (km)")
# plt.ylabel("Temperature ($^o$ C)")
# plt.legend()
# plt.grid()
# plt.show()

####
#### plot the pressure profile
####
# plt.scatter(uAlt/1000,uPress/1013.25*100,marker='o',label="Flight Computer (inside box)")
# plt.scatter(aW5/1000,pW5/1013.25*100,marker='o',label="APRS (inside box)")
# plt.scatter(aK5/1000,pK5/1013.25*100,marker='o',label="APRS (outside box)")
# plt.xlabel("Altitude (km)")
# plt.ylabel("Pressure (\% of STP)")
# plt.yscale('log')
# plt.legend()
# plt.grid()
# plt.show()

####
#### plot the speed profile
####
# plt.scatter(uAlt/1000,uSpeed,marker='o',label="Flight Computer (inside box)")
# plt.scatter(aW5/1000,sW5,marker='o',label="APRS (inside box)")
# plt.scatter(aK5/1000,sK5,marker='o',label="APRS (outside box)")
# plt.xlabel("Altitude (km)")
# plt.ylabel("Speed (kph)")
# plt.axhline(160.934,0,1)
# plt.text(16,164,"100 mph")
# plt.legend()
# plt.grid()
# plt.show()

####
#### plot the flight models
####
# xp = np.linspace(2800,10332,10000)
# plt.plot(xp,ascModel(xp))
# plt.plot(xp,ascModel_K(xp))
# plt.plot(xp,ascModel_W(xp))
# plt.grid()
# plt.show()


####
#### plot the ascent/descent rate
####
# tmask = ((flightTime>2800)&(flightTime<12800))
# delT_FC = np.diff(flightTime[tmask])
# delT_K = np.diff(tK5)
# delT_W = np.diff(tW5)
# delA_FC = np.abs(np.diff(uAlt[tmask]))
# delA_K = np.abs(np.diff(aK5))
# delA_W = np.abs(np.diff(aW5))

# tmask_ascFC = ((flightTime>2800)&(flightTime<10332))
# tmask_ascK = (tK5<10332)
# tmask_ascW = (tW5<10332)
# ascAve_FC = np.average(np.abs(np.diff(uAlt[tmask_ascFC])) / np.diff(flightTime[tmask_ascFC]))
# ascAve_K = np.average(np.abs(np.diff(aK5[tmask_ascK])) / np.diff(tK5[tmask_ascK]))
# ascAve_W = np.average(np.abs(np.diff(aW5[tmask_ascW])) / np.diff(tW5[tmask_ascW]))
# ascAve = np.average([ascAve_FC,ascAve_K,ascAve_W])
# errAve = np.std([ascAve_FC,ascAve_K,ascAve_W])
# #print(ascAve,errAve)

# plt.scatter(flightTime[tmask][1:],delA_FC/delT_FC,marker='o',label="Flight Computer (inside box)")
# plt.scatter(tW5[1:],delA_W/delT_W,marker='o',label="APRS (inside box)")
# plt.scatter(tK5[1:],delA_K/delT_K,marker='o',label="APRS (outside box)")
# plt.fill_between(np.linspace(2800,10332,1000),ascAve-errAve,ascAve+errAve,color='red',alpha=0.5)
# plt.hlines(ascAve,2800.,10332.,color='black',lw=2)
# plt.xlabel("Flight Time (seconds)")
# plt.ylabel("Ascent/Descent Rate (m/s)")
# plt.grid()
# plt.show()


####
#### plot the cosmic ray rate as a function of altitude
####

## ASCENT
rateMask_asc = ((binned_time*60>2800)&(binned_time*60<10332))
cosmicTime_asc = binned_time[rateMask_asc]
cosmicRate_asc = binned_count_rate[rateMask_asc]
cosmicRateErr_asc = binned_count_rate_err[rateMask_asc]
cosmicAlt_asc = np.zeros(len(cosmicRate_asc))
cosmicAltErr_asc = np.zeros(len(cosmicRate_asc))
## combine flight models for a single point
for i in range(len(cosmicAlt_asc)):
    cosmicAlt_asc[i] = np.average([ascModel(cosmicTime_asc[i]*60),ascModel_K(cosmicTime_asc[i]*60),ascModel_W(cosmicTime_asc[i]*60)])
    cosmicAltErr_asc[i] = np.std([ascModel(cosmicTime_asc[i]*60),ascModel_K(cosmicTime_asc[i]*60),ascModel_W(cosmicTime_asc[i]*60)])
#gRateMask = ((adjGtime>2800)&(adjGtime<10332))
gRateMask_asc = ((gTime>2800)&(gTime<10332))
gCosmicTime_asc = gTime[gRateMask_asc]
gCosmicRate_asc = gRate[gRateMask_asc]
gCosmicRateErr_asc = gErr[gRateMask_asc]
gCosmicAlt_asc = np.zeros(len(gCosmicRate_asc))
gCosmicAltErr_asc = np.zeros(len(gCosmicRate_asc))
## combine flight models for a single point
for i in range(len(gCosmicAlt_asc)):
    gCosmicAlt_asc[i] = np.average([ascModel(gCosmicTime_asc[i]),ascModel_K(gCosmicTime_asc[i]),ascModel_W(gCosmicTime_asc[i])])
    gCosmicAltErr_asc[i] = np.std([ascModel(gCosmicTime_asc[i]),ascModel_K(gCosmicTime_asc[i]),ascModel_W(gCosmicTime_asc[i])])

## DESCENT
rateMask_dsc = ((binned_time*60>10332)&(binned_time*60<12800))
cosmicTime_dsc = binned_time[rateMask_dsc]
cosmicRate_dsc = binned_count_rate[rateMask_dsc]
cosmicRateErr_dsc = binned_count_rate_err[rateMask_dsc]
cosmicAlt_dsc = np.zeros(len(cosmicRate_dsc))
cosmicAltErr_dsc = np.zeros(len(cosmicRate_dsc))
## combine flight models for a single point
for i in range(len(cosmicAlt_dsc)):
    cosmicAlt_dsc[i] = np.average([desModel(cosmicTime_dsc[i]*60),desModel_K(cosmicTime_dsc[i]*60),desModel_W(cosmicTime_dsc[i]*60)])
    cosmicAltErr_dsc[i] = np.std([desModel(cosmicTime_dsc[i]*60),desModel_K(cosmicTime_dsc[i]*60),desModel_W(cosmicTime_dsc[i]*60)])
#gRateMask2 = ((adjGtime>10332)&(adjGtime<12800))
gRateMask_dsc = ((gTime>10332)&(gTime<12800))
gCosmicTime_dsc = gTime[gRateMask_dsc]
gCosmicRate_dsc = gRate[gRateMask_dsc]
gCosmicRateErr_dsc = gErr[gRateMask_dsc]
gCosmicAlt_dsc = np.zeros(len(gCosmicRate_dsc))
gCosmicAltErr_dsc = np.zeros(len(gCosmicRate_dsc))
## combine flight models for a single point
for i in range(len(gCosmicAlt_dsc)):
    gCosmicAlt_dsc[i] = np.average([desModel(gCosmicTime_dsc[i]),desModel_K(gCosmicTime_dsc[i]),desModel_W(gCosmicTime_dsc[i])])
    gCosmicAltErr_dsc[i] = np.std([desModel(gCosmicTime_dsc[i]),desModel_K(gCosmicTime_dsc[i]),desModel_W(gCosmicTime_dsc[i])])

## GC rates are in cpm
## normalize to detector area
gCosmicRate_asc = gCosmicRate_asc/60./_GC_AREA_
gCosmicRateErr_asc = gCosmicRateErr_asc/60./_GC_AREA_
gCosmicRate_dsc = gCosmicRate_dsc/60./_GC_AREA_
gCosmicRateErr_dsc = gCosmicRateErr_dsc/60./_GC_AREA_
## CW rates already in cnts/sec
## normalize to detector area
cosmicRate_asc = cosmicRate_asc/_CW_AREA_
cosmicRateErr_asc = cosmicRateErr_asc/_CW_AREA_
cosmicRate_dsc = cosmicRate_dsc/_CW_AREA_
cosmicRateErr_dsc = cosmicRateErr_dsc/_CW_AREA_

## fit data with polynomial
rateFit_asc = np.polyfit(cosmicAlt_asc,cosmicRate_asc,_FITORDER_)
rateModel_asc = np.poly1d(rateFit_asc)
rateFit_dsc = np.polyfit(cosmicAlt_dsc,cosmicRate_dsc,_FITORDER_)
rateModel_dsc = np.poly1d(rateFit_dsc)
gRateFit_asc = np.polyfit(gCosmicAlt_asc,gCosmicRate_asc,_FITORDER_)
gRateModel_asc = np.poly1d(gRateFit_asc)
gRateFit_dsc = np.polyfit(gCosmicAlt_dsc,gCosmicRate_dsc,_FITORDER_)
gRateModel_dsc = np.poly1d(gRateFit_dsc)
rXp = np.linspace(500,32000,10000)

####
#### find maximum value from derivatives of the fitted polynomials
####
crit_asc = rateModel_asc.deriv().r
r_crit_asc = crit_asc[crit_asc.imag==0].real
test_asc = rateModel_asc.deriv(2)(r_crit_asc) ## make sure the curvature is correct
x_max_asc = r_crit_asc[test_asc<0][0]
y_max_asc = rateModel_asc(x_max_asc)

crit_dsc = rateModel_dsc.deriv().r
r_crit_dsc = crit_dsc[crit_dsc.imag==0].real
test_dsc = rateModel_dsc.deriv(2)(r_crit_dsc)
x_max_dsc = r_crit_dsc[test_dsc<0][0] ## make sure the curvature is correct
y_max_dsc = rateModel_dsc(x_max_dsc)

gCrit_asc = gRateModel_asc.deriv().r
gr_crit_asc = gCrit_asc[gCrit_asc.imag==0].real
gTest_asc = gRateModel_asc.deriv(2)(gr_crit_asc)
gx_max_asc = gr_crit_asc[gTest_asc<0][0] ## make sure the curvature is correct
gy_max_asc = gRateModel_dsc(gx_max_asc)

gCrit_dsc = gRateModel_dsc.deriv().r
gr_crit_dsc = gCrit_dsc[gCrit_dsc.imag==0].real
gTest_dsc = gRateModel_dsc.deriv(2)(gr_crit_dsc)
gx_max_dsc = gr_crit_dsc[gTest_dsc<0][0] ## make sure the curvature is correct
gy_max_dsc = gRateModel_dsc(gx_max_dsc)
print(x_max_asc,y_max_asc,x_max_dsc,y_max_dsc,gx_max_asc,gy_max_asc,gx_max_dsc,gy_max_dsc)


####
#### print some results to file for later
####
# with open('CW_results.csv','w') as f:
#     for i in range(len(cosmicAlt_asc)):
#         f.write("%f,%f,%f,%f\n"%(cosmicAlt_asc[i]/1000,cosmicRate_asc[i],cosmicAltErr_asc[i]/1000,cosmicRateErr_asc[i]))
#     for i in range(len(cosmicAlt_dsc)):
#         f.write("%f,%f,%f,%f\n"%(cosmicAlt_dsc[i]/1000,cosmicRate_dsc[i],cosmicAltErr_dsc[i]/1000,cosmicRateErr_dsc[i]))
# with open('GC_results.csv','w') as f:
#     for i in range(len(gCosmicAlt_asc)):
#         f.write("%f,%f,%f,%f\n"%(gCosmicAlt_asc[i]/1000,gCosmicRate_asc[i],gCosmicAltErr_asc[i]/1000,gCosmicRateErr_asc[i]))
#     for i in range(len(gCosmicAlt_dsc)):
#         f.write("%f,%f,%f,%f\n"%(gCosmicAlt_dsc[i]/1000,gCosmicRate_dsc[i],gCosmicAltErr_dsc[i]/1000,gCosmicRateErr_dsc[i]))

####
#### plot the results
####
plt.errorbar(cosmicAlt_asc/1000,cosmicRate_asc,xerr=cosmicAltErr_asc/1000,yerr=cosmicRateErr_asc,fmt='o',color='blue')
#plt.plot(rXp,rateModel_asc(rXp))
plt.errorbar(cosmicAlt_dsc/1000,cosmicRate_dsc,xerr=cosmicAltErr_dsc/1000,yerr=cosmicRateErr_dsc,fmt='o',color='black')
#plt.plot(rXp,rateModel_dsc(rXp))
plt.errorbar(gCosmicAlt_asc/1000,gCosmicRate_asc,xerr=gCosmicAltErr_asc/1000,yerr=gCosmicRateErr_asc,fmt='o',color='red')
#plt.plot(rXp/1000,gRateModel_asc(rXp))
plt.errorbar(gCosmicAlt_dsc/1000,gCosmicRate_dsc,xerr=gCosmicAltErr_dsc/1000,yerr=gCosmicRateErr_dsc,fmt='o',color='green')
#plt.errorbar(gCosmicAlt_dsc/1000,gCosmicRate_dsc,xerr=0,yerr=0,fmt='o')
plt.axvline(x_max_asc/1000,color='blue')
plt.axvline(x_max_dsc/1000,color='black')
plt.axvline(gx_max_asc/1000,color='red')
plt.axvline(gx_max_dsc/1000,color='green')
plt.text(12,0.3,"CW Ascent Maximum Rate of %.2f @ %.2f km"%(y_max_asc,x_max_asc/1000),fontsize=16)
plt.text(12,0.25,"CW Descent Maximum Rate of %.2f @ %.2f km"%(y_max_dsc,x_max_dsc/1000),fontsize=16)
plt.text(12,0.01,"Geiger Ascent Maximum Rate of %.2f @ %.2f km"%(gy_max_asc,gx_max_asc/1000),fontsize=16)
plt.text(12,0.05,"Geiger Descent Maximum Rate of %.2f @ %.2f km"%(gy_max_dsc,gx_max_dsc/1000),fontsize=16)
plt.text(3,0.1,"PRELIMINARY",fontsize=75,color='gray',alpha=0.5,rotation=45)
# plt.scatter(cosmicAlt_asc,cosmicRate_asc - rateModel_asc(cosmicAlt_asc),marker='.')
# plt.scatter(cosmicAlt_dsc,cosmicRate_dsc - rateModel_dsc(cosmicAlt_dsc),marker='.')
plt.xlabel("Altitude (km)")
plt.ylabel("Cosmic Ray Rate (cnts/sec/cm$^2$)")
#plt.xlim(0,35)
plt.grid()
plt.show()


####
#### plot rate vs time
####
plt.errorbar(cosmicTime_asc*60,cosmicRate_asc,xerr=cosmicAltErr_asc/1000,yerr=cosmicRateErr_asc,fmt='o',color='blue')
#plt.plot(rXp,rateModel_asc(rXp))
plt.errorbar(cosmicTime_dsc*60,cosmicRate_dsc,xerr=cosmicAltErr_dsc/1000,yerr=cosmicRateErr_dsc,fmt='o',color='blue')
#plt.plot(rXp,rateModel_dsc(rXp))
plt.errorbar(gCosmicTime_asc,gCosmicRate_asc,xerr=gCosmicAltErr_asc/1000,yerr=gCosmicRateErr_asc,fmt='o',color='black')
#plt.plot(rXp/1000,gRateModel_asc(rXp))
plt.errorbar(gCosmicTime_dsc,gCosmicRate_dsc,xerr=gCosmicAltErr_dsc/1000,yerr=gCosmicRateErr_dsc,fmt='o',color='black')
plt.xlabel("Time (seconds)")
plt.ylabel("Cosmic Ray Rate (cnts/sec/cm$^2$)")
plt.grid()
plt.show()
#

## Q.E.F.